In [2]:
import os
import pyspark

# 1. Clear broken background environment variables
os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

# 2. Detect your PySpark version to ensure perfect compatibility
spark_version = pyspark.__version__
is_spark4 = spark_version.startswith("4")

# 3. Bundle Iceberg + AWS + Kafka packages together
if is_spark4:
    working_packages = (
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.10.0,"
        f"org.apache.spark:spark-sql-kafka-0-10_2.13:{spark_version}"
    )
else:
    working_packages = (
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
        f"org.apache.spark:spark-sql-kafka-0-10_2.12:{spark_version}"
    )

# 4. Start the session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("CDC-Bronze") \
  .config("spark.jars.packages", working_packages) \
  .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.lakehouse.type", "rest") \
  .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181") \
  .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
  .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000") \
  .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true") \
  .config("spark.sql.defaultCatalog", "lakehouse") \
  .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")
print(f"Success! Spark {spark_version} loaded with Iceberg and Kafka.")

Success! Spark 4.1.0 loaded with Iceberg and Kafka.


In [3]:
raw = spark.read \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "kafka:9092") \
  .option("subscribe", "dbserver1.public.customers") \
  .option("startingOffsets", "earliest") \
  .load()

# To verify it worked, run this in the next cell:
raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [ ]:
from pyspark.sql import functions as F

# Filter out tombstone records (null value) first
raw_filtered = raw.filter(F.col("value").isNotNull())

bronze_df = raw_filtered.select(
  F.col("topic"),
  F.col("partition").alias("kafka_partition"),
  F.col("offset").alias("kafka_offset"),
  F.col("timestamp").alias("kafka_timestamp"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("long").alias("ts_ms"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.name").alias("after_name"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.email").alias("after_email"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.country").alias("after_country"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
)

bronze_df.writeTo("lakehouse.cdc.bronze_customers").createOrReplace()

In [ ]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.bronze_customers").show()
spark.sql("""
  SELECT op, after_id, after_name, after_email, ts_ms
  FROM lakehouse.cdc.bronze_customers LIMIT 5
""").show(truncate=False)

In [ ]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_customers (
    id INT, name STRING, email STRING, country STRING, last_updated_ms BIGINT
  ) USING iceberg
""")

In [ ]:
from pyspark.sql.window import Window

# Use COALESCE because for deletes, after_id is null but before_id has the key
bronze_with_key = bronze_df.withColumn(
  "entity_id", F.coalesce(F.col("after_id"), F.col("before_id"))
)

w = Window.partitionBy("entity_id").orderBy(F.col("ts_ms").desc())
deduped = bronze_with_key \
  .filter(F.col("op").isNotNull()) \
  .withColumn("rn", F.row_number().over(w)) \
  .filter("rn = 1").drop("rn")

In [ ]:
deduped.createOrReplaceTempView("cdc_batch")

spark.sql("""
  MERGE INTO lakehouse.cdc.silver_customers AS t
  USING cdc_batch AS s
  ON t.id = s.entity_id
  
  WHEN MATCHED AND s.op = 'd' THEN DELETE
  
  WHEN MATCHED AND s.op IN ('c','u','r') THEN UPDATE SET
    t.name = s.after_name, t.email = s.after_email,
    t.country = s.after_country, t.last_updated_ms = s.ts_ms

  WHEN NOT MATCHED AND s.op != 'd' THEN INSERT
    (id, name, email, country, last_updated_ms)
    VALUES (s.after_id, s.after_name, s.after_email, s.after_country, s.ts_ms)

""")

In [ ]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT * FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5").show(truncate=False)

In [ ]:
spark.sql("SELECT count(*) AS silver_customers_count FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT count(*) AS silver_trips_count FROM lakehouse.taxi.silver_trips").show()